# CS383: Data Science and Machine Learning
## Lecture 5 — Exploratory Data Analysis and Foundational Statistics

**Guiding question:** before you trust a chart or a model, can you describe what's actually in your
data — its center, its spread, its shape, and how (if at all) its variables relate to each other?

Today stays entirely on the NYC 311 dataset you already know from Lectures 1-4 — no new dataset to get
oriented in, so the focus can stay on the statistics and EDA habits themselves. `resolution_time_hours`
is the running example throughout, and you'll see this exact column again in Lecture 7, when it becomes
a real regression target.

---
**Live in-class version.** Type along at each `__________` blank — everything else is filled in so class
time stays on the new syntax, not on retyping boilerplate.

---

### Optional: review these concepts interactively

Before diving into Part 1 — or any time you want a refresher — this self-paced page walks through
mean/median/mode, standard deviation & IQR, distribution shape and skew, and correlation vs. causation.
Click-through practice with instant feedback, no coding required, about 10-15 minutes.

**[Open the Statistics Review Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect05/stats_review_activity.html)**

Good to revisit again before Lecture 7, where several of these ideas (outlier-sensitivity, skew, the
mean-vs-median gap) come back as reasons *why* a regression model behaves the way it does.

---

## Part 1 — Descriptive Statistics

Before any chart, a few numbers already tell you a lot about a dataset: where its center is, how spread
out it is, and where the middle 50% of it lives.

### Warm-up: two classes' worth of test scores

Before turning to real (much messier) NYC 311 data, let's see mean, median, standard deviation, and
outliers on something small enough to check by eye: two sections of the same 24-student exam, scored out
of 100.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(383)

class_a_scores = rng.normal(78, 6, 24).round().astype(int).clip(0, 100)
class_b_typical = rng.normal(78, 6, 21).round().astype(int).clip(0, 100)
class_b_scores = np.__________([class_b_typical, [20, 15, 25]])  # 3 students who had a genuinely rough exam

print("Class A:", sorted(class_a_scores.tolist()))
print("Class B:", sorted(class_b_scores.tolist()))

In [ ]:
scores_df = pd.DataFrame({"Class A": class_a_scores, "Class B": class_b_scores})
sns.__________(data=scores_df)
plt.ylabel("Exam score")
plt.title("Two classes, same exam")
plt.show()

In [ ]:
print(f"Class A -- mean: {class_a_scores.mean():.1f}, median: {np.__________(class_a_scores):.1f}")
print(f"Class B -- mean: {class_b_scores.mean():.1f}, median: {np.median(class_b_scores):.1f}")

Class A's mean and median land close together — a fairly symmetric class. Class B's median is nearly
identical to Class A's — the *typical* student in Class B did just as well. But Class B's mean is
noticeably lower. That gap is entirely the work of three students who had a genuinely rough exam. If you
only reported the mean, you'd conclude Class B underperformed Class A; the median tells a more
representative story about most of the class.

In [ ]:
print(f"Class A -- std: {class_a_scores.__________():.1f}")
print(f"Class B -- std: {class_b_scores.std():.1f}")

Class B's standard deviation is more than 3x Class A's — not because most students in Class B are
inconsistent, but because a small number of very different scores stretch the spread a lot. Std doesn't
distinguish "everyone is a little different" from "almost everyone is similar, plus a few outliers" — for
that, you need to actually look, which is exactly what the box plot above already showed you before you
computed a single number.

In [ ]:
q1, q3 = np.quantile(class_b_scores, [0.25, 0.75])
iqr = q3 - q1
lower_bound = q1 - __________ * iqr

flagged = sorted(class_b_scores[class_b_scores < lower_bound].tolist())
print(f"Class B IQR lower bound: {lower_bound:.1f}")
print(f"Flagged as outliers: {flagged}")

The same 1.5×IQR rule Part 3 formalizes below catches exactly the three planted low scores, and nothing
else — on a small, fully-visible dataset like this, you can immediately confirm the rule did the right
thing by eye. That's the payoff of starting small: once the same rule gets applied to thousands of real,
messy rows starting in Part 1's real data below, you already know what it's supposed to be catching.

---

With mean, median, standard deviation, and outliers established on a dataset small enough to check by
eye, the rest of this lecture applies the exact same tools to real NYC 311 data — messier, real, and not
something you can just eyeball.

### Setup — NYC 311

Same dataset and live-pull-with-fallback pattern as Lectures 1-4, with `resolution_time_hours` computed
the same way as Lecture 4.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df.head()

In [ ]:
complaints_df[["resolution_time_hours", "hour_filed"]].__________()

### Mean, median, and mode

In [ ]:
resolution_times = complaints_df["resolution_time_hours"].dropna()

manual_mean = resolution_times.sum() / len(resolution_times)
print(f"Manual mean:     {manual_mean:.2f} hours")
print(f"pandas .mean():  {resolution_times.__________():.2f} hours")

The **mean** is the sum divided by the count — the familiar "average." It's easy to compute, but it's
pulled around by extreme values: a handful of complaints that took an unusually long time to resolve drag
the mean upward even if most complaints close quickly. Keep that in mind — Lecture 6's `StandardScaler`
centers every feature on its mean, so a skewed column's "center" isn't necessarily where most of your
data actually sits.

*Reference figure — for visual intuition, not something you need to memorize.*

![Four different distributions, each with its mean marked by a red line](images/mean-of-distributions.png)

The mean isn't just a formula — it's the balance point of a distribution. All four curves above have
very different shapes and centers, but the red line always lands on the same spot: the point where the
distribution balances left-to-right.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

In [ ]:
sorted_times = resolution_times.sort_values().reset_index(drop=True)
mid = len(sorted_times) // 2
if len(sorted_times) % 2 == 0:
    manual_median = (sorted_times[mid - 1] + sorted_times[mid]) / 2
else:
    manual_median = sorted_times[mid]

print(f"Manual median:    {manual_median:.2f} hours")
print(f"pandas .median(): {resolution_times.__________():.2f} hours")

The **median** is the middle value once the data is sorted — half the complaints resolve faster than
it, half slower. Unlike the mean, a handful of extremely slow complaints barely move it. Mean and median
noticeably disagree here — that's a skew signal, and it's exactly the shape you'll see again in
Lecture 7, when `resolution_time_hours` becomes a regression target and that same skew is part of why a
plain linear fit struggles with it.

In [ ]:
manual_mode = complaints_df["complaint_type"].value_counts().idxmax()
print(f"Manual mode:    {manual_mode}")
print(f"pandas .mode(): {complaints_df['complaint_type'].__________().tolist()}")

# Mode is often more useful for categorical columns than numeric ones:
print("\nFull breakdown, most to least common:")
print(complaints_df["complaint_type"].value_counts())

The **mode** is simply the most frequent value. For a continuous numeric column like
`resolution_time_hours`, it's rarely the most informative statistic — but for a categorical column like
`complaint_type`, it's often exactly what you want ("what's the most common outcome?"). This particular
breakdown happens to be fairly balanced across complaint types — but that won't always be true of a
categorical column. If one ever were lopsided, and especially if it were your prediction *target* rather
than just a feature, that imbalance would matter a lot: accuracy alone can be misleading when one
category dominates, which is exactly what Lecture 9 covers.

### Variance and standard deviation

In [ ]:
mean_time = resolution_times.mean()
squared_devs = (resolution_times - mean_time) ** 2
manual_variance = squared_devs.sum() / (len(resolution_times) - 1)
manual_std = manual_variance ** 0.5

print(f"Manual variance: {manual_variance:.2f}")
print(f"pandas .var():   {resolution_times.__________():.2f}")
print(f"Manual std dev:  {manual_std:.2f}")
print(f"pandas .std():   {resolution_times.std():.2f}")

`squared_devs` here is `(resolution_time - mean) ** 2` computed across every row at once — variance is
just the *average* of those squared deviations, and standard deviation is variance's square root, back
in the original units (hours). One detail worth being precise about: pandas divides by `n - 1`, not `n`
(a correction for estimating from a sample rather than a full population) — that's why the manual formula
above uses `len(resolution_times) - 1` too, so the two agree. Standard deviation is also exactly what
`StandardScaler` divides by when it rescales a feature in Lecture 6 — a column's spread here is literally
the number that scaling later depends on.

*Reference figure — for visual intuition, not something you need to memorize.*

![A bell curve with 1, 2, and 3 standard deviation bands shaded around the mean](images/std-dev-bands.png)

Standard deviation isn't just a number — it's a unit of distance from the mean. For data shaped like
this curve, roughly 68% of values fall within 1 standard deviation of the mean (the darkest band), about
95% within 2, and about 99.7% within 3. This is sometimes called the *68-95-99.7 rule*, and it's why a
value more than 2-3 standard deviations out already looks unusual before you even apply a formal outlier
rule.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

### Quartiles

In [ ]:
q1, q2, q3 = resolution_times.__________([0.25, 0.5, 0.75])
print(f"Q1 (25th percentile):        {q1:.1f} hours")
print(f"Q2 (50th percentile/median): {q2:.1f} hours")
print(f"Q3 (75th percentile):        {q3:.1f} hours")

**Quartiles** split sorted data into four equal-count groups. Q2 is just the median again; Q1 and Q3
mark where the bottom and top quarters begin. The distance between them, `Q3 - Q1`, is the
**interquartile range (IQR)** — the width of the "typical" middle 50% of the data, which Part 3 uses to
flag outliers. It matters beyond EDA, too: Lecture 7's regression models minimize *squared* error, so a
single value far outside the IQR gets squared right along with everything else — meaning one extreme,
unflagged outlier can distort a fitted line far more than its single row should.

---

## Part 2 — Distributions

A single number (the mean) can't tell you the *shape* of your data — whether it's symmetric, skewed, or
has multiple clusters. For that, you need to look at the whole distribution.

In [ ]:
plt.hist(resolution_times, bins=__________, edgecolor="white")
plt.xlabel("Resolution time (hours)")
plt.ylabel("Number of complaints")
plt.title("Distribution of 311 complaint resolution time")
plt.show()

A **histogram** groups values into bins and counts how many fall in each. Notice the shape: a longer
tail stretching toward longer resolution times. That's a **right skew** — it's exactly why the mean came
out higher than the median in Part 1: a relatively small number of very slow complaints pull the mean up,
but don't move the median much. Hold onto this exact shape — Lecture 7 reaches for a log-transform
specifically because of skew that looks like this.

### The same plot, with Seaborn

In [ ]:
sns.histplot(data=complaints_df, x="resolution_time_hours", kde=__________)
plt.title("Distribution of 311 complaint resolution time")
plt.show()

**Seaborn** is a plotting library built directly on top of Pandas DataFrames and Matplotlib — you hand it
a `DataFrame` and column names instead of raw arrays, and it comes with better default styling. The
`kde=True` overlay adds a smoothed curve tracing the distribution's shape, which makes skew easier to see
at a glance than bars alone.

*Reference figure — for visual intuition, not something you need to memorize.*

![A distribution with two separate peaks](images/bimodal-distribution.png)

Not every distribution looks like a single bell curve. This one has two peaks — it's **bimodal**. That
shape is often a clue that two different groups got mixed into one column (e.g., "amount of oil in the
car" might really be two populations: cars just topped off, and cars way overdue for a change). A
histogram is what would reveal this; a mean or median alone would hide it completely.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

---

## Part 3 — Outlier Detection with IQR

A common, principled rule for flagging outliers uses the interquartile range from Part 1: anything more
than 1.5×IQR below Q1 or above Q3 is flagged.

In [ ]:
q1, q3 = resolution_times.quantile([0.25, 0.75])
iqr = q3 __________ q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"IQR: {iqr:.1f} hours")
print(f"Bounds: [{lower_bound:.1f}, {upper_bound:.1f}] hours")

1.5×IQR is a convention, not a law of nature — it's a widely-used rule of thumb (originating with the
box plot) for "unusually far from the middle 50%," not a statistical test proving something is wrong
with a data point.

In [ ]:
sns.__________(y=resolution_times)
plt.ylabel("Resolution time (hours)")
plt.title("311 complaint resolution time, with outliers")
plt.show()

A **box plot** draws the box from Q1 to Q3 (the IQR), a line at the median, "whiskers" out to the last
point within the 1.5×IQR bounds, and individual dots for everything beyond that — exactly the outliers
you just computed.

In [ ]:
outliers = complaints_df[
    (complaints_df["resolution_time_hours"] < lower_bound)
    __________ (complaints_df["resolution_time_hours"] > upper_bound)
]
print(f"{len(outliers):,} complaints flagged as outliers ({len(outliers) / len(resolution_times):.1%})")
outliers[["complaint_type", "borough", "resolution_time_hours"]].sort_values(
    "resolution_time_hours", ascending=False
).head()

Careful: "outlier" doesn't automatically mean "delete." A complaint flagged here might be a data-entry
problem — a `closed_date` logged incorrectly — or it might be a genuinely, importantly neglected case
that a journalist or oversight body would very much want to know about. IQR flags a point as *worth a
second look*; what you do next is a judgment call, not something the formula decides for you.

---

## Part 4 — Correlation vs. Causation

Everything up to this point — mean, median, std, quartiles, the histogram, the IQR outlier rule — looked
at one column at a time. That's **univariate** analysis. The moment you ask whether two columns move
together, you've stepped into **bivariate** analysis (two variables); ask that question across three or
more columns at once and it's **multivariate**.

Do any of these variables move together — and when they do (or don't), can you tell whether that's a
real relationship or something else entirely?

In [ ]:
sns.scatterplot(data=complaints_df, x="hour_filed", y="resolution_time_hours", alpha=__________)
plt.title("Resolution time vs. hour filed")
plt.show()

No obvious trend — the cloud looks close to a formless blob. That's a real, honest result worth taking
seriously rather than glossing over: whatever drives how long a complaint takes to resolve, the hour it
was filed doesn't seem to be much of it.

In [ ]:
complaints_df[["resolution_time_hours", "hour_filed"]].__________().round(3)

In [ ]:
sns.heatmap(
    complaints_df[["resolution_time_hours", "hour_filed"]].corr(),
    annot=__________, cmap="coolwarm", vmin=-1, vmax=1,
)
plt.title("Correlation matrix")
plt.show()

`.corr()` computes the **correlation coefficient** between every pair of numeric columns — a number from
-1 (perfectly opposite) to +1 (perfectly aligned), with 0 meaning no linear relationship. A heatmap just
makes a whole matrix of these easier to scan at once than a table of numbers. Here it's close to 0 either
way — matching what the scatter plot already showed, and a preview of what Lecture 7 digs into properly
when this same column becomes a regression target.

### Anscombe's Quartet

*Reference figure — for visual intuition, not something you need to memorize.*

![Four scatter plots that all share the same mean, variance, and correlation coefficient, but look
completely different](images/anscombe-quartet-panels.png)

These four datasets have *identical* summary statistics: same mean, same variance, same correlation
coefficient, even the same best-fit line. Here they are as four separate scatter plots — a clean linear
relationship, a curve, a near-perfect line thrown off by one point, and a vertical cluster thrown off by
one point.

*Figure adapted from Andrew Glassner's* Deep Learning: A Visual Approach *(No Starch Press), released
under the MIT license.*

![The same four datasets, superimposed on one plot](images/anscombe-quartet-superimposed.png)

Superimposed, the difference is obvious. If you had only looked at `.corr()` or `.describe()` for each of
these four datasets, you would have concluded they were basically the same. This is exactly why Parts 1-3
of this lecture never stopped at numbers alone — a chart can show you something a summary statistic
hides entirely.

In [ ]:
anscombe_x = {
    "I":   [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "II":  [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "III": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5],
    "IV":  [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
}
anscombe_y = {
    "I":   [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68],
    "II":  [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74],
    "III": [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73],
    "IV":  [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
}

for name in anscombe_x:
    x = np.array(anscombe_x[name])
    y = np.array(anscombe_y[name])
    print(f"Dataset {name}: mean(x)={x.mean():.2f}  mean(y)={y.mean():.2f}  "
          f"var(x)={x.var(ddof=1):.2f}  var(y)={y.var(ddof=1):.2f}  "
          f"corr={np.corrcoef(x, y)[0,1]:.3f}")

Not an approximation — these four wildly different-looking datasets really do share the same mean,
variance, and correlation, to two or three decimal places. The only way to tell them apart is to actually
plot them.

### Correlation is not causation

The classic warning example: ice cream sales and drowning deaths rise and fall together throughout the
year, strongly correlated — but ice cream doesn't cause drowning. Both are driven by a third factor: hot
weather brings more people to both ice cream stands and swimming pools. That hidden third factor is
called a **confounder**.

### A confounder you might expect — checked against real data

You might expect complaint volume to move together across boroughs: a slow weekend citywide, a busy
weekday citywide, the same weather affecting everyone at once. Let's check.

In [ ]:
complaints_df["created_day"] = complaints_df["created_date"].dt.date
daily_by_borough = complaints_df.groupby(["created_day", "borough"]).size().__________(fill_value=0)

sns.heatmap(daily_by_borough.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation of daily complaint counts, across boroughs")
plt.show()

Every pair of boroughs here comes out close to zero — not the shared citywide pattern you might have
expected. Before concluding boroughs' complaint volumes really are unrelated day to day, though, think
about how this dataset was built: the shared snapshot samples a fixed number of complaints per month,
spread evenly on purpose, so that every lecture gets a representative mix of months regardless of when
in the semester it runs. That design choice is exactly what would flatten out a real day-of-week or
weather effect, if one exists in the live data. This is itself a real EDA habit worth keeping: before
trusting a correlation — or a lack of one — ask how the data was collected, not just what the number
says.

---

## Part 5 — Choosing the Right Chart

The chart you reach for follows directly from how many variables the question involves — univariate,
bivariate, or multivariate:

| Question | Variables | Chart |
|---|---|---|
| How many rows fall into each category? | Univariate (1 categorical) | Bar chart |
| What's the shape of one numeric column's distribution? | Univariate (1 numeric) | Histogram |
| How does one numeric column compare across categories? | Bivariate (1 numeric + 1 categorical) | Box plot |
| Is there a relationship between two numeric columns? | Bivariate (2 numeric) | Scatter plot |
| How do many numeric columns relate to each other at once? | Multivariate (3+ numeric) | Heatmap (of `.corr()`) |

The most common mistake isn't picking a "wrong" chart technically — it's picking a chart that
*technically* works but obscures the point you're actually trying to make. Part 6 puts this into
practice.

---

## Part 6 — Putting It Together: Telling This as a Story

Same 311 data, but the goal shifts entirely now: turn what you found in Parts 1-4 into one finding you
could actually explain to someone who's never opened a notebook. This is exactly what Assignment 2 asks
you to do.

### Technical storytelling, for Assignment 2

Assignment 2 asks for an EDA and visualization report written for a non-technical audience. A simple
framework:

1. **Lead with the finding** — the sentence a busy reader would actually remember, not the method.
2. **One supporting chart** — the smallest chart that proves the finding, not every chart you made along
   the way.
3. **No jargon** — "IQR," "correlation coefficient," and "groupby" mean nothing to this reader. Say what
   you found in plain language instead.
4. **State the "so what"** — why should this reader care? What would they do differently knowing this?

In [ ]:
avg_resolution_by_borough = (
    complaints_df.groupby("borough")["resolution_time_hours"].mean().sort_values(ascending=__________)
)

avg_resolution_by_borough.plot(kind="barh")
plt.xlabel("Average hours to resolve a complaint")
plt.title("How long does it take to resolve a 311 complaint, by borough?")
plt.gca().invert_yaxis()
plt.show()

print(avg_resolution_by_borough.round(1))

**Applying the framework:**

*(Fill in the bracketed borough names using your own output from the chart above — the live-pull data
changes run to run, so the answer isn't fixed.)*

> On average, complaints in [slowest borough] take the longest to resolve — noticeably longer than in
> [fastest borough]. If you've filed a 311 complaint and it feels like it's taking forever, where you
> live may genuinely be part of why.

Notice what that summary leaves out: no mention of `groupby`, no p-values, no mention of the underlying
dataset size. One chart, one finding, one reason to care.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect05_eda_statistics_exercise.ipynb`.

---

## Part 7 — Cheat Sheet

| Task | Code |
|---|---|
| Mean / median / mode | `.mean()` / `.median()` / `.mode()` |
| Variance / standard deviation | `.var()` / `.std()` |
| Quartiles | `.quantile([0.25, 0.5, 0.75])` |
| IQR outlier bounds | `q1 - 1.5*iqr`, `q3 + 1.5*iqr` |
| Histogram | `sns.histplot(data=df, x="col", kde=True)` |
| Box plot | `sns.boxplot(y=df["col"])` |
| Scatter plot | `sns.scatterplot(data=df, x="a", y="b", hue="c")` |
| Correlation matrix | `df[cols].corr()` |
| Heatmap | `sns.heatmap(df[cols].corr(), annot=True)` |

---

## Part 8 — Key Terms

- **Mean**: the sum of values divided by the count; sensitive to extreme values.
- **Median**: the middle value once sorted; resistant to extreme values.
- **Mode**: the most frequently occurring value.
- **Variance / standard deviation**: measures of how spread out values are around the mean.
- **Quartile / IQR**: the values splitting data into four equal-count groups; `Q3 - Q1` is the
  interquartile range.
- **Skew**: an asymmetric distribution, with a longer tail on one side (often visible as mean ≠ median).
- **Outlier**: a value unusually far from the rest of the data, by some rule (e.g., 1.5×IQR beyond
  Q1/Q3).
- **Correlation coefficient**: a number from -1 to +1 describing how strongly two numeric variables move
  together linearly.
- **Confounder**: a hidden third factor that drives two variables to correlate without either causing
  the other.